## Modelo XGBoost

### Carga de librerías

In [2]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import log_loss
import xgboost as xgb
from pathlib import Path

### Carga de datos

In [10]:
ROOT = Path("..")
ruta_t1 = ROOT / "data" / "processed" / "temporada1_con_nulos.parquet"
temporada1 = pl.read_parquet(ruta_t1).to_pandas()

Vamos a chequear las columnas que tiene el dataset de la temporada 2. Las columnas a incluir en el modelo XGBoost van a ser todas las que contenga el dataset de la temporada 2 más las que se calculan a partir de ellas (*feature engineering*)

La única diferencia entre los archivos .parquet de las temporadas 1 y 2 son las variables *swing* y *description* (variable respuesta) y todas las que calculamos en el *feature engineering*.

## Primer modelo - Excluímos variables del sensor

### Definición de variables predictoras y respuesta

In [4]:
columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
columnas_contaminadas = [
    'sz_top', 'sz_bot', 'sz_mid', 'strike_zone_size', 'relative_height', 'pitch_location',
    'is_strike_zone', 'is_shadow_zone', 'distance_to_corner'
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(columns=[col for col in todas_excluidas if col in temporada1.columns])
respuesta = temporada1['swing']

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype('category')

print(f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras.")

El dataset tiene 709852 filas y 15 variables predictoras.


### Configuración del modelo XGBoost

In [8]:
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    enable_categorical=True, # Avisa que hay columnas categóricas
    tree_method='hist',      # Algoritmo de histogramas, rápido para datasets grandes
    random_state=714
)

Vamos a definir una serie de hiperparámetros distintos y vamos a probar solo 10 al azar. Queda para otro momento el analizar otras técnicas de selección de hiperparámetros.

In [9]:
param_dist = {
    'max_depth': [3, 5, 7],               # Profundidad del árbol
    'learning_rate': [0.01, 0.05, 0.1],   # Qué tan rápido aprende
    'n_estimators': [100, 300, 500],      # Cantidad de árboles
    'subsample': [0.8, 1.0],              # Porcentaje de filas a usar por árbol
    'colsample_bytree': [0.8, 1.0]        # Porcentaje de columnas a usar por árbol
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) # Usamos 3 folds para que sea más rápido
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=10, # límite de combinaciones de hiperparámetros a probar
    scoring='neg_log_loss', # Scikit-learn busca maximizar, por eso usa LogLoss negativo
    cv=skf,
    verbose=1, # Te va mostrando el progreso en la consola
    random_state=569
)

random_search.fit(explicativas, respuesta)
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor LogLoss estimado: {-random_search.best_score_: .4f}")

mejor_modelo = random_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Mejores hiperparámetros: {'subsample': 1.0, 'n_estimators': 500, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Mejor LogLoss estimado:  0.4474


### Predicción para Kaggle

Primero, leemos los datos limpios de la temporada 2.

In [10]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"

temporada2= pl.read_parquet(ruta_t2)

In [11]:
temporada2 = temporada2.to_pandas()

# Guardamos los IDs para el archivo final (Kaggle siempre pide el ID)
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test para que tenga EXACTAMENTE las mismas columnas que usamos para entrenar
X_test = temporada2.drop(columns=[col for col in todas_excluidas if col in temporada2.columns])

# Hay un error de categoría 1-3. Lo convertimos en nulo.
X_test['count'] = X_test['count'].replace('1-3', np.nan)

# Pasamos las variables necesarias a categóricas, con las mismas categorías que el training set
columnas_texto = explicativas.select_dtypes(include=['category']).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(X_test[col], categories=explicativas[col].cat.categories)


# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': probabilidades_swing
})

In [12]:
# Guardamos el archivo
ruta_prediccion = ROOT / "reports" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

## Segundo modelo - Variables de sensor modificadas

In [11]:
HALF_PLATE_CM = 21.59

# Calculamos la zona histórica solo con los datos de entrenamiento
zona_promedio = temporada1.groupby('batter')[['sz_top', 'sz_bot']].mean().reset_index()
zona_promedio = zona_promedio.rename(columns={'sz_top': 'sz_top_mean', 'sz_bot': 'sz_bot_mean'})

# Cruzamos los promedios al dataset
temporada1 = temporada1.merge(zona_promedio, on='batter', how='left')

In [12]:
# Calculamos la geometría estática
temporada1['strike_zone_size_curada'] = temporada1['sz_top_mean'] - temporada1['sz_bot_mean']
temporada1['relative_height_curada'] = (temporada1['plate_z'] - temporada1['sz_bot_mean']) / temporada1['strike_zone_size_curada']
temporada1['relative_x'] = temporada1['plate_x'] / HALF_PLATE_CM

In [13]:
# Variables espaciales seguras
temporada1['is_strike_zone_curada'] = (
    (temporada1['plate_x'] >= -HALF_PLATE_CM) & 
    (temporada1['plate_x'] <= HALF_PLATE_CM) & 
    (temporada1['relative_height_curada'] >= 0) & 
    (temporada1['relative_height_curada'] <= 1)
).astype(int)

In [14]:
condicion_zona_t1 = temporada1['is_strike_zone_curada'] == 1
distancia_adentro_t1 = np.sqrt(
    (1 - temporada1['relative_x'].abs())**2 + 
    np.minimum(temporada1['relative_height_curada'], 1 - temporada1['relative_height_curada'])**2
)
temporada1['distance_to_corner_curada'] = np.where(condicion_zona_t1, distancia_adentro_t1, 0.0)

In [15]:
# Eliminamos las variables base y las espaciales contaminadas originales
columnas_excluir = ['pitch_id', 'batter', 'pitcher', 'description', 'swing']
columnas_contaminadas = [
    'sz_top', 'sz_bot', 'sz_mid', 'strike_zone_size', 'relative_height', 'pitch_location',
    'is_strike_zone', 'is_shadow_zone', 'distance_to_corner',
    'sz_top_mean', 'sz_bot_mean', 'relative_x' # Sacamos las intermedias para que no hagan ruido
]

todas_excluidas = columnas_excluir + columnas_contaminadas

explicativas = temporada1.drop(columns=[col for col in todas_excluidas if col in temporada1.columns])
respuesta = temporada1['swing']

# A XGBoost hay que definirle cuáles son las variables categóricas
columnas_texto = explicativas.select_dtypes(include=['object', 'string']).columns
for col in columnas_texto:
    explicativas[col] = explicativas[col].astype('category')

print(f"El dataset tiene {explicativas.shape[0]} filas y {explicativas.shape[1]} variables predictoras.")

El dataset tiene 709852 filas y 18 variables predictoras.


In [16]:
xgb_base = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    enable_categorical=True, 
    tree_method='hist',      
    random_state=714
)

In [17]:
param_dist = {
    'max_depth': [3, 5, 7],               
    'learning_rate': [0.01, 0.05, 0.1],   
    'n_estimators': [100, 300, 500],      
    'subsample': [0.8, 1.0],              
    'colsample_bytree': [0.8, 1.0]        
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42) 
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=10, 
    scoring='neg_log_loss', 
    cv=skf,
    verbose=1, 
    random_state=569
)

random_search.fit(explicativas, respuesta)
print(f"Mejores hiperparámetros: {random_search.best_params_}")
print(f"Mejor LogLoss estimado: {-random_search.best_score_: .4f}")

mejor_modelo = random_search.best_estimator_

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Mejores hiperparámetros: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Mejor LogLoss estimado:  0.4382


### Predicción para Kaggle

In [18]:
ruta_t2 = ROOT / "data" / "processed" / "temporada2_limpio.parquet"
temporada2 = pl.read_parquet(ruta_t2).to_pandas()

# Usamos el zona_promedio que calculamos con la temporada 1
temporada2 = temporada2.merge(zona_promedio, on='batter', how='left')

Al modelo lo entramos con el promedio de 'sz_top' y 'sz_bot' por bateador. El problema es que el dataset de la temporada 2 puede tener jugadores nuevos para los que no vamos a contar con estas medidas en la temporada 1. Vamos a ver cuántos son estos nuevos jugadores.

In [22]:
# Filtramos el dataframe quedándonos solo con las filas que no cruzaron (jugadores nuevos)
filas_nuevos = temporada2[temporada2['sz_top_mean'].isna()]

# Conteos de filas
nulos_t2 = len(filas_nuevos)
total_filas_t2 = len(temporada2)

# Conteos de individuos únicos
jugadores_nuevos = filas_nuevos['batter'].nunique()
total_jugadores_t2 = temporada2['batter'].nunique()

print(f"Pitcheos de bateadores nuevos: {nulos_t2} de {total_filas_t2} ({nulos_t2/total_filas_t2*100:.2f}%)")
print(f"Cantidad de bateadores nuevos: {jugadores_nuevos} de {total_jugadores_t2} bateadores totales en la T2")

Pitcheos de bateadores nuevos: 81435 de 708310 (11.50%)
Cantidad de bateadores nuevos: 178 de 693 bateadores totales en la T2


A los bateadores nuevos los vamos a imputar con medidas estándar.

In [23]:
temporada2['sz_top_mean'] = temporada2['sz_top_mean'].fillna(3.5 * 30.48)
temporada2['sz_bot_mean'] = temporada2['sz_bot_mean'].fillna(1.5 * 30.48)

temporada2['strike_zone_size_curada'] = temporada2['sz_top_mean'] - temporada2['sz_bot_mean']
temporada2['relative_height_curada'] = (temporada2['plate_z'] - temporada2['sz_bot_mean']) / temporada2['strike_zone_size_curada']
temporada2['relative_x'] = temporada2['plate_x'] / HALF_PLATE_CM

temporada2['is_strike_zone_curada'] = (
    (temporada2['plate_x'] >= -HALF_PLATE_CM) & 
    (temporada2['plate_x'] <= HALF_PLATE_CM) & 
    (temporada2['relative_height_curada'] >= 0) & 
    (temporada2['relative_height_curada'] <= 1)
).astype(int)

condicion_zona_t2 = temporada2['is_strike_zone_curada'] == 1
distancia_adentro_t2 = np.sqrt(
    (1 - temporada2['relative_x'].abs())**2 + 
    np.minimum(temporada2['relative_height_curada'], 1 - temporada2['relative_height_curada'])**2
)
temporada2['distance_to_corner_curada'] = np.where(condicion_zona_t2, distancia_adentro_t2, 0.0)

In [ ]:
# Guardamos los IDs para el archivo final
pitch_ids_t2 = temporada2['pitch_id']

# Filtramos X_test
X_test = temporada2.drop(columns=[col for col in todas_excluidas if col in temporada2.columns])

# Alineamos las variables categóricas con las de entrenamiento
columnas_texto = explicativas.select_dtypes(include=['category']).columns
for col in columnas_texto:
    X_test[col] = pd.Categorical(X_test[col], categories=explicativas[col].cat.categories)

# Predecimos las probabilidades
probabilidades_swing = mejor_modelo.predict_proba(X_test)[:, 1]

# Armamos y guardamos el DataFrame final
predicciones = pl.DataFrame({
    'pitch_id': pitch_ids_t2,
    'swing_probability': probabilidades_swing
})

ruta_prediccion = ROOT / "reports" / "validacion.parquet"
predicciones.write_parquet(ruta_prediccion)

Como este modelo dio una predicción sobre el dataset de la temporada 2 ligeramente mejor al primer modelo XGBoost, vamos a deternos a analizar cuáles fueron las principales variables que utilizó para clasificar.

In [26]:
importancias = mejor_modelo.feature_importances_

df_importancias = pd.DataFrame({
    'variable': explicativas.columns,
    'importancia': importancias
})

df_importancias = df_importancias.sort_values(by='importancia', ascending=False)

print(" == TOP 10 VARIABLES MÁS IMPORTANTES ==")
print(df_importancias.head(10).to_string(index=False))

 == TOP 10 VARIABLES MÁS IMPORTANTES ==
                 variable  importancia
    is_strike_zone_curada     0.614801
                  strikes     0.195638
distance_to_corner_curada     0.089200
   relative_height_curada     0.016116
                  plate_x     0.014162
                    balls     0.010070
                 p_throws     0.008329
                    pfx_x     0.008082
                    count     0.008057
                    pfx_z     0.006970


- `is_strike_zone_curada` (61.78%): la decisión de hacer swing depende casi en su totalidad de si la pelota va a la zona buena o no.

- `strikes` (19.6%): un bateador no se comporta igual cuando tiene 0 strikes que cuendo tiene 2 y tiene que defender el plato para no quedar eliminado. Podría interpretarse como un efecto "presión" del juego.

- `distance_to_corner_curada` (8.9%): el modelo detectó que las pelotas que llegan en las esquinas de la zona de strike generan una duda o un comportamiento distinto en el bateador respecto a las pelotas que van cómodas al centro.